In [94]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("Data")
RAW_DIR = DATA_DIR / "Raw"
PROCESSED_DIR = DATA_DIR / "Processed"
for directory in (RAW_DIR, PROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def load_csv_datasets(raw_dir):
    csv_paths = sorted(raw_dir.glob("*.csv"))
    datasets = {path.stem: pd.read_csv(path) for path in csv_paths}
    return datasets, csv_paths


raw_csv_datasets, raw_csv_paths = load_csv_datasets(RAW_DIR)
expected_csv_stems = {path.stem for path in raw_csv_paths}
loaded_csv_stems = set(raw_csv_datasets)
missing_csv_stems = sorted(expected_csv_stems - loaded_csv_stems)
if missing_csv_stems:
    raise RuntimeError(f"Missing CSV loads for: {', '.join(missing_csv_stems)}")

governance_workbook_path = RAW_DIR / "Worldwide_Governance_Indicators.xlsx"
if not governance_workbook_path.exists():
    raise FileNotFoundError(f"Missing workbook: {governance_workbook_path}")

dfx = pd.read_excel(governance_workbook_path)

# Legacy aliases used by later cells
# Keep these names stable so the rest of the notebook does not need to change.
df_rd = raw_csv_datasets["OECD_RD_GDP_2000_2025"]
df3 = raw_csv_datasets["OECD_Productivity_Variation_2000_2025"]
df4 = raw_csv_datasets["OECD_Tertiary_Education_2000_2025"]
df5 = raw_csv_datasets["OECD_Unemployment_Rate_2000_2025"]
df6 = raw_csv_datasets["OECD_Inflation_CPI_2000_2025"]
dft = raw_csv_datasets.get("OECD_Trust_Institutions_2000_2025")
dfe = raw_csv_datasets.get("Eurostat_Energy_Import_Dependency_2000_2025", raw_csv_datasets["Global_Energy_Dependency_0_100"] )
dfg = raw_csv_datasets["OECD_GDP_Growth_2000_2025"]
df_ge = raw_csv_datasets["Global_Energy_Dependency_0_100"]
df_eur = raw_csv_datasets["Eurostat_Public_Debt_GDP_2000_2025"]
df_gd = raw_csv_datasets["Global_Public_Debt_GDP_2000_2025"]

raw_file_inventory = pd.DataFrame(
    [
        {
            "file_name": path.name,
            "dataset_name": path.stem,
            "file_type": "csv",
            "rows": int(raw_csv_datasets[path.stem].shape[0]),
            "columns": int(raw_csv_datasets[path.stem].shape[1]),
        }
        for path in raw_csv_paths
    ]
)

raw_file_inventory = pd.concat(
    [
        raw_file_inventory,
        pd.DataFrame(
            [
                {
                    "file_name": governance_workbook_path.name,
                    "dataset_name": governance_workbook_path.stem,
                    "file_type": "xlsx",
                    "rows": int(dfx.shape[0]),
                    "columns": int(dfx.shape[1]),
                }
            ]
        ),
    ],
    ignore_index=True,
 )

raw_file_inventory = raw_file_inventory.sort_values(["file_type", "file_name"]).reset_index(drop=True)

print("Loaded CSV datasets:")
print(", ".join(sorted(raw_csv_datasets)))
print("Loaded workbook:")
print(governance_workbook_path.name)
print("Inventory ready in memory")

Loaded CSV datasets:
Eurostat_Public_Debt_GDP_2000_2025, Global_Energy_Dependency_0_100, Global_Public_Debt_GDP_2000_2025, OECD_GDP_Growth_2000_2025, OECD_Inflation_CPI_2000_2025, OECD_Productivity_Variation_2000_2025, OECD_RD_GDP_2000_2025, OECD_Tertiary_Education_2000_2025, OECD_Unemployment_Rate_2000_2025
Loaded workbook:
Worldwide_Governance_Indicators.xlsx
Inventory ready in memory


In [25]:
from pathlib import Path

PROCESSED_DIR = Path("Data") / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd

In [26]:
import pandas as pd

In [27]:
import pandas as pd

In [28]:
print(raw_file_inventory.sort_values(["file_type", "file_name"]).to_string(index=False))

                                file_name                          dataset_name file_type  rows  columns
   Eurostat_Public_Debt_GDP_2000_2025.csv    Eurostat_Public_Debt_GDP_2000_2025       csv   702        3
       Global_Energy_Dependency_0_100.csv        Global_Energy_Dependency_0_100       csv  3151        3
     Global_Public_Debt_GDP_2000_2025.csv      Global_Public_Debt_GDP_2000_2025       csv   788        3
            OECD_GDP_Growth_2000_2025.csv             OECD_GDP_Growth_2000_2025       csv  1314        3
         OECD_Inflation_CPI_2000_2025.csv          OECD_Inflation_CPI_2000_2025       csv  1362        3
OECD_Productivity_Variation_2000_2025.csv OECD_Productivity_Variation_2000_2025       csv  4247        3
                OECD_RD_GDP_2000_2025.csv                 OECD_RD_GDP_2000_2025       csv  1127        3
    OECD_Tertiary_Education_2000_2025.csv     OECD_Tertiary_Education_2000_2025       csv  1003        3
     OECD_Unemployment_Rate_2000_2025.csv      OECD_Une

In [29]:
dfe

,Country,Year,Value
0,AGO,2000,0.000000
1,AGO,2001,0.000000
2,AGO,2002,0.000000
3,AGO,2003,0.000000
4,AGO,2004,0.000000
...,...,...,...
3146,ZWE,2018,23.868661
3147,ZWE,2019,21.200162
3148,ZWE,2020,17.352023
3149,ZWE,2021,9.322581


In [30]:
# pycountry is expected to be available in the notebook environment

In [31]:
import pycountry

def convert_iso2_to_iso3(code):
    if pd.isna(code):
        return None
    code = str(code).strip().upper()
    if not code:
        return None
    country = pycountry.countries.get(alpha_2=code)
    if country is None:
        try:
            country = pycountry.countries.lookup(code)
        except LookupError:
            return None
    return country.alpha_3

In [32]:
dfe["Country"] = dfe["Country"].apply(convert_iso2_to_iso3)

In [33]:
dfe.head(30)

,Country,Year,Value
0,AGO,2000,0.000000
1,AGO,2001,0.000000
2,AGO,2002,0.000000
3,AGO,2003,0.000000
4,AGO,2004,0.000000
5,AGO,2005,0.000000
6,AGO,2006,0.000000
7,AGO,2007,0.000000
8,AGO,2008,0.000000
9,AGO,2009,0.000000


In [34]:
# Check inconsistent text
if "Country" in dfe.columns:
    dfe = dfe.copy()
    dfe["Country"] = dfe["Country"].astype("string").str.lower().str.strip()

In [35]:
dfe

,Country,Year,Value
0,ago,2000,0.000000
1,ago,2001,0.000000
2,ago,2002,0.000000
3,ago,2003,0.000000
4,ago,2004,0.000000
...,...,...,...
3146,zwe,2018,23.868661
3147,zwe,2019,21.200162
3148,zwe,2020,17.352023
3149,zwe,2021,9.322581


In [ ]:
import pandas as pd
import pycountry

csv_dataset_names = sorted(raw_csv_datasets.keys())
files = [f"{dataset_name}.csv" for dataset_name in csv_dataset_names]


# =========================
# COUNTRY CODE FIX
# =========================
def iso2_to_iso3(code):
    if pd.isna(code):
        return None
    code = str(code).strip().upper()
    if not code:
        return None
    country = pycountry.countries.get(alpha_2=code)
    if country is None:
        try:
            country = pycountry.countries.lookup(code)
        except LookupError:
            return None
    return country.alpha_3


# =========================
# CLEANING FUNCTION
# =========================
def clean_dataset(df, fix_iso2=False):
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()

    required_columns = {"country", "year", "value"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise KeyError(f"Missing required columns: {', '.join(sorted(missing_columns))}")

    df = df[["country", "year", "value"]]
    df["country"] = df["country"].astype("string").str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    if fix_iso2:
        df["country"] = df["country"].apply(iso2_to_iso3)

    df["country"] = df["country"].fillna("MISSING_COUNTRY")
    df = df.dropna(subset=["year"] )
    df["year"] = df["year"].astype(int)
    df = df.drop_duplicates().sort_values(["country", "year"]).reset_index(drop=True)

    df["value"] = df.groupby("country", dropna=False)["value"].transform(
        lambda x: x.interpolate(limit_direction="both").ffill().bfill()
    )

    return df


# =========================
# LOAD + CLEAN
# =========================
datasets = {}

for file in files:
    dataset_key = file.replace(".csv", "")
    df = raw_csv_datasets[dataset_key].copy()

    # Detect Eurostat dataset by file name
    if "Eurostat" in file:
        datasets[dataset_key] = clean_dataset(df, fix_iso2=True)
    else:
        datasets[dataset_key] = clean_dataset(df, fix_iso2=False)


# =========================
# PIVOT (SAFE VERSION)
# =========================
pivot_datasets = {}

for name, df in datasets.items():
    pivot_datasets[name] = df.pivot_table(
        index="year",
        columns="country",
        values="value",
        aggfunc="mean"
    ).sort_index()

final_clean = None
dfs_long = []

for name, df in pivot_datasets.items():
    temp = df.reset_index().melt(
        id_vars="year",
        var_name="country",
        value_name="value"
    )
    temp["indicator"] = name
    dfs_long.append(temp)

df_all = pd.concat(dfs_long, ignore_index=True)
final_df = df_all.pivot_table(
    index=["year", "country"],
    columns="indicator",
    values="value"
).reset_index()

final_clean = final_df.copy()
numeric_cols = final_clean.select_dtypes(include="number").columns
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.interpolate(limit_direction="both")
 )
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.fillna(x.mean())
)

final_clean

indicator,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,ARG,NaN,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,7.561
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3710,2025,SWE,35.1,27.752405,46.363,1.543167,0.679869,332.353013,3.559558,51.826313,8.398
3711,2025,TUR,NaN,71.992860,28.557,3.604625,34.881160,91.185172,1.461902,26.944220,8.712
3712,2025,USA,NaN,0.000000,123.186,2.106338,2.949525,90.335094,3.444858,50.705994,4.022
3713,2025,USMCA,NaN,NaN,NaN,1.931042,NaN,NaN,NaN,NaN,NaN


In [37]:
df_ge

,Country,Year,Value
0,AGO,2000,0.000000
1,AGO,2001,0.000000
2,AGO,2002,0.000000
3,AGO,2003,0.000000
4,AGO,2004,0.000000
...,...,...,...
3146,ZWE,2018,23.868661
3147,ZWE,2019,21.200162
3148,ZWE,2020,17.352023
3149,ZWE,2021,9.322581


In [38]:
#merging the datasets based on the year
# STEP 1: Convert each pivot dataset to LONG format
dfs_long = []

for name, df in pivot_datasets.items():
    
    temp = df.reset_index().melt(
        id_vars="year",
        var_name="country",
        value_name="value"
    )
    
    temp["indicator"] = name  # keep track of dataset source
    
    dfs_long.append(temp)


# STEP 2: Concatenate everything into one dataset
df_all = pd.concat(dfs_long, ignore_index=True)


# STEP 3: Pivot to final clean panel format
final_df = df_all.pivot_table(
    index=["year", "country"],
    columns="indicator",
    values="value"
).reset_index()




#Fill intelligently, NaN does NOT mean bad data, it means “data not available for that country-year-indicator”
final_clean = final_df.copy()

numeric_cols = final_clean.select_dtypes(include="number").columns

# 1. interpolate (time structure)
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.interpolate(limit_direction="both")
)

# 2. country mean
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.fillna(x.mean())
)




final_clean

indicator,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,ARG,NaN,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,7.561
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3710,2025,SWE,35.1,27.752405,46.363,1.543167,0.679869,332.353013,3.559558,51.826313,8.398
3711,2025,TUR,NaN,71.992860,28.557,3.604625,34.881160,91.185172,1.461902,26.944220,8.712
3712,2025,USA,NaN,0.000000,123.186,2.106338,2.949525,90.335094,3.444858,50.705994,4.022
3713,2025,USMCA,NaN,NaN,NaN,1.931042,NaN,NaN,NaN,NaN,NaN


In [39]:

final_clean

indicator,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,ARG,NaN,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,7.561
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3710,2025,SWE,35.1,27.752405,46.363,1.543167,0.679869,332.353013,3.559558,51.826313,8.398
3711,2025,TUR,NaN,71.992860,28.557,3.604625,34.881160,91.185172,1.461902,26.944220,8.712
3712,2025,USA,NaN,0.000000,123.186,2.106338,2.949525,90.335094,3.444858,50.705994,4.022
3713,2025,USMCA,NaN,NaN,NaN,1.931042,NaN,NaN,NaN,NaN,NaN


In [95]:
report_path = PROCESSED_DIR / "final_dataset_complete.csv"
report_final_clean = globals().get("final_clean")
if report_final_clean is None:
    if not report_path.exists():
        raise FileNotFoundError(f"Missing cleaned panel export: {report_path}")
    report_final_clean = pd.read_csv(report_path)

report_final_clean = report_final_clean.copy()
report_final_clean["country"] = report_final_clean["country"].fillna("MISSING_COUNTRY")

missing_by_country_long = (
    report_final_clean
    .melt(id_vars=["country"], var_name="column", value_name="value")
    .assign(is_missing=lambda frame: frame["value"].isna())
    .groupby(["country", "column"], as_index=False)["is_missing"]
    .sum()
    .sort_values(["country", "column"] )
    .reset_index(drop=True)
 )

missing_country_rows = report_final_clean[report_final_clean["country"].eq("MISSING_COUNTRY")].copy()
missing_by_country_long

,country,column,is_missing
0,AGO,Eurostat_Public_Debt_GDP_2000_2025,23
1,AGO,Global_Energy_Dependency_0_100,0
2,AGO,Global_Public_Debt_GDP_2000_2025,23
3,AGO,OECD_GDP_Growth_2000_2025,23
4,AGO,OECD_Inflation_CPI_2000_2025,23
...,...,...,...
1615,ZWE,OECD_Productivity_Variation_2000_2025,23
1616,ZWE,OECD_RD_GDP_2000_2025,23
1617,ZWE,OECD_Tertiary_Education_2000_2025,23
1618,ZWE,OECD_Unemployment_Rate_2000_2025,23


In [ ]:
# duplicate report export removed

In [42]:
# pycountry is expected to be available in the notebook environment

In [43]:
#Create conversion function
import pycountry

def country_to_iso3(country_name):
    if pd.isna(country_name):
        return None
    country_name = str(country_name).strip()
    if not country_name:
        return None
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except LookupError:
        return None



In [48]:
dfx = pd.read_excel(RAW_DIR / "Worldwide_Governance_Indicators.xlsx")

In [ ]:
# Keep the cleaned governance workbook in memory only; do not export an intermediate CSV

In [50]:
dfx = dfx[dfx["Year"] >= 2000]
dfx

,ID variable (economy code/ gov. dimension/ year),Economy (name),Economy (code),Region,Income classification,Year,Governance dimension,Number of sources,Governance estimate (approx. -2.5 to +2.5),Standard error (estimate),...,OBI mean,PIA mean,PRS mean,RSF mean,VAB mean,VDM mean,WBS mean,WCY mean,WJP mean,WMO mean
403,ADOva2000,Andorra,ADO,NaN,NaN,2000,va,3,1.541361,0.301021,...,..,..,..,..,..,..,..,..,..,1
404,AFGva2000,Afghanistan,AFG,South Asia,Low income,2000,va,4,-2.297846,0.245080,...,..,..,..,..,..,0.245567,..,..,..,0.0625
405,AGOva2000,Angola,AGO,Sub-Saharan Africa,Lower middle income,2000,va,7,-1.678503,0.185910,...,..,..,0.333333,..,..,0.320267,..,..,..,0.125
406,ALBva2000,Albania,ALB,Europe & Central Asia,Upper middle income,2000,va,6,-0.524937,0.204243,...,..,..,0.75,..,..,0.460222,..,..,..,0.375
407,AREva2000,United Arab Emirates,ARE,Middle East & North Africa,High income,2000,va,6,-0.843748,0.193985,...,..,..,0.583333,..,..,0.335233,..,..,..,0.6875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,XKXva2024,Kosovo,XKX,Europe & Central Asia,Upper middle income,2024,va,9,0.159588,0.156446,...,..,0.5,..,0.5273,..,0.607289,..,..,0.578919,..
5440,YEMva2024,"Yemen, Rep.",YEM,Middle East & North Africa,Low income,2024,va,9,-1.697182,0.178875,...,0,0.1,0.416667,0.3145,..,0.3571,..,..,..,..
5441,ZAFva2024,South Africa,ZAF,Sub-Saharan Africa,Upper middle income,2024,va,13,0.607280,0.147152,...,0.82713,..,0.833333,0.7571,..,0.650411,..,0.314754,0.619057,..
5442,ZMBva2024,Zambia,ZMB,Sub-Saharan Africa,Lower middle income,2024,va,14,-0.407569,0.142237,...,0.339358,0.3,0.791667,0.5733,..,0.5176,..,..,0.421383,..


In [51]:
dfx["Economy (name)"] = dfx["Economy (name)"].apply(country_to_iso3)

In [52]:
dfx

,ID variable (economy code/ gov. dimension/ year),Economy (name),Economy (code),Region,Income classification,Year,Governance dimension,Number of sources,Governance estimate (approx. -2.5 to +2.5),Standard error (estimate),...,OBI mean,PIA mean,PRS mean,RSF mean,VAB mean,VDM mean,WBS mean,WCY mean,WJP mean,WMO mean
403,ADOva2000,AND,ADO,NaN,NaN,2000,va,3,1.541361,0.301021,...,..,..,..,..,..,..,..,..,..,1
404,AFGva2000,AFG,AFG,South Asia,Low income,2000,va,4,-2.297846,0.245080,...,..,..,..,..,..,0.245567,..,..,..,0.0625
405,AGOva2000,AGO,AGO,Sub-Saharan Africa,Lower middle income,2000,va,7,-1.678503,0.185910,...,..,..,0.333333,..,..,0.320267,..,..,..,0.125
406,ALBva2000,ALB,ALB,Europe & Central Asia,Upper middle income,2000,va,6,-0.524937,0.204243,...,..,..,0.75,..,..,0.460222,..,..,..,0.375
407,AREva2000,ARE,ARE,Middle East & North Africa,High income,2000,va,6,-0.843748,0.193985,...,..,..,0.583333,..,..,0.335233,..,..,..,0.6875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,XKXva2024,None,XKX,Europe & Central Asia,Upper middle income,2024,va,9,0.159588,0.156446,...,..,0.5,..,0.5273,..,0.607289,..,..,0.578919,..
5440,YEMva2024,None,YEM,Middle East & North Africa,Low income,2024,va,9,-1.697182,0.178875,...,0,0.1,0.416667,0.3145,..,0.3571,..,..,..,..
5441,ZAFva2024,ZAF,ZAF,Sub-Saharan Africa,Upper middle income,2024,va,13,0.607280,0.147152,...,0.82713,..,0.833333,0.7571,..,0.650411,..,0.314754,0.619057,..
5442,ZMBva2024,ZMB,ZMB,Sub-Saharan Africa,Lower middle income,2024,va,14,-0.407569,0.142237,...,0.339358,0.3,0.791667,0.5733,..,0.5176,..,..,0.421383,..


In [53]:
dfx.columns

Index(['ID variable (economy code/ gov. dimension/ year)', 'Economy (name)',
       'Economy (code)', 'Region', 'Income classification', 'Year',
       'Governance dimension', 'Number of sources',
       'Governance estimate (approx. -2.5 to +2.5)',
       'Standard error (estimate)',
       'Lower threshold (90% conf. int. estimate)',
       'Upper threshold (90% conf. int. estimate)', 'Governance score (0-100)',
       'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)',
       'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean',
       'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean',
       'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean',
       'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean',
       'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean',
       'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM mean', 'WBS mean',
       'WCY mean', 'WJP mean', 'WM

In [54]:
dfx = dfx.drop(['ID variable (economy code/ gov. dimension/ year)',
       'Economy (code)', 'Region', 'Income classification',
       'Governance dimension', 'Number of sources',
       'Governance estimate (approx. -2.5 to +2.5)',
       'Standard error (estimate)',
       'Lower threshold (90% conf. int. estimate)',
       'Upper threshold (90% conf. int. estimate)',
       'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)',
       'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean',
       'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean',
       'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean',
       'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean',
       'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean',
       'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM mean', 'WBS mean',
       'WCY mean', 'WJP mean', 'WMO mean'], axis=1)

In [55]:
dfx

,Economy (name),Year,Governance score (0-100)
403,AND,2000,83.826902
404,AFG,2000,16.631208
405,AGO,2000,27.471257
406,ALB,2000,47.661534
407,ARE,2000,42.081555
...,...,...,...
5439,None,2024,59.642428
5440,None,2024,27.144335
5441,ZAF,2024,67.478144
5442,ZMB,2024,49.715769


In [56]:
dfx.rename(columns={"Economy (name)": "country"}, inplace=True)
dfx.rename(columns={"Year": "year"}, inplace=True)

In [57]:
#merging
final_merged = pd.merge(
    final_clean,
    dfx,
    on=["country", "year"],
    how="inner"
)

In [58]:
final_merged

,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025,Governance score (0-100)
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.471257
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.661534
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.081555
3,2000,ARG,NaN,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,7.561,64.590212
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.781340
...,...,...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,66.4,48.902182,69.318,1.731397,1.965627,48.513447,2.162628,34.637478,3.697,75.351985
2857,2024,SWE,34.2,27.752405,46.363,1.017797,2.835817,332.353013,3.559558,51.826313,8.398,87.089942
2858,2024,TUR,NaN,71.992860,28.557,3.327623,58.506450,91.185172,1.461902,26.944220,8.712,37.689790
2859,2024,USA,NaN,0.000000,123.186,2.793187,2.949525,90.335094,3.444858,50.705994,4.022,72.373172


In [59]:
missing_rows = final_clean[final_clean.isnull().any(axis=1)]
missing_rows.head(30)

indicator,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,ARG,NaN,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,7.561000
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2000,AUS,NaN,0.000000,12.765,3.359670,4.457435,52.464076,1.469756,27.475746,6.288000
7,2000,AZE,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2000,BEN,NaN,28.097710,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,2000,BGD,NaN,17.650915,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,2000,BGR,70.7,46.864953,48.474,NaN,10.316260,11.549349,0.495167,27.498056,16.919000


In [67]:
final_merged = final_merged.rename(columns={"OECD_GDP_Growth_2000_2025": "R&D expenditure (%GDP)"})
final_merged = final_merged.rename(columns={"Governance score (0-100)": "Worldwide_Governance_Indicators"})
final_merged = final_merged.rename(columns={"Global_Public_Debt_GDP_2000_2025": "Public_Debt (%GDP)"})
final_merged = final_merged.rename(columns={"Global_Energy_Dependency_0_100": "Energy_Import_Dependency"})
final_merged = final_merged.rename(columns={"OECD_Tertiary_Education_2000_2025": "Tertiary_education_attainment"})

In [68]:
final_df1 = final_merged[
    [
        "year",
        "country",
        "R&D expenditure (%GDP)",
        "Worldwide_Governance_Indicators",
        "Public_Debt (%GDP)",
        "Energy_Import_Dependency",
        "Tertiary_education_attainment"
     
    ]
]

In [70]:
final_merged = final_merged.rename(columns={"OECD_Inflation_CPI_2000_2025": "Inflation_Rate"})
final_merged = final_merged.rename(columns={"OECD_Unemployment_Rate_2000_2025": "Unemployment_Variation"})
final_merged = final_merged.rename(columns={"Governance score (0-100)": "Worldwide_Governance_Indicators"})
final_merged = final_merged.rename(columns={"OECD_Productivity_Variation_2000_2025": "Productivity_Variation"})
final_merged = final_merged.rename(columns={"R&D expenditure (%GDP)": "GDP_Growth"})
final_df2 = final_merged[
    [
        "year",
        "country",
        "Inflation_Rate",
        "Unemployment_Variation",
        "Worldwide_Governance_Indicators",
        "Productivity_Variation",
        "GDP_Growth"
        
    ]
]

In [71]:
final_df1

,year,country,R&D expenditure (%GDP),Worldwide_Governance_Indicators,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment
0,2000,AGO,NaN,27.471257,NaN,0.000000,NaN
1,2000,ALB,NaN,47.661534,NaN,46.518154,NaN
2,2000,ARE,NaN,42.081555,NaN,0.000000,NaN
3,2000,ARG,-0.788999,64.590212,NaN,0.000000,16.711861
4,2000,ARM,NaN,46.781340,NaN,71.191142,NaN
...,...,...,...,...,...,...,...
2856,2024,SVN,1.731397,75.351985,69.318,48.902182,34.637478
2857,2024,SWE,1.017797,87.089942,46.363,27.752405,51.826313
2858,2024,TUR,3.327623,37.689790,28.557,71.992860,26.944220
2859,2024,USA,2.793187,72.373172,123.186,0.000000,50.705994


In [72]:
final_df2

,year,country,Inflation_Rate,Unemployment_Variation,Worldwide_Governance_Indicators,Productivity_Variation,GDP_Growth
0,2000,AGO,NaN,NaN,27.471257,NaN,NaN
1,2000,ALB,NaN,NaN,47.661534,NaN,NaN
2,2000,ARE,NaN,NaN,42.081555,NaN,NaN
3,2000,ARG,34.277230,7.561,64.590212,NaN,-0.788999
4,2000,ARM,NaN,NaN,46.781340,NaN,NaN
...,...,...,...,...,...,...,...
2856,2024,SVN,1.965627,3.697,75.351985,48.513447,1.731397
2857,2024,SWE,2.835817,8.398,87.089942,332.353013,1.017797
2858,2024,TUR,58.506450,8.712,37.689790,91.185172,3.327623
2859,2024,USA,2.949525,4.022,72.373172,90.335094,2.793187


In [74]:
final_df3 = final_merged[
    [
        "year",
        "country",
        "Public_Debt (%GDP)",
        "Energy_Import_Dependency",
        "Tertiary_education_attainment",
        "Inflation_Rate",
        "Unemployment_Variation",
        "Worldwide_Governance_Indicators",
        "Productivity_Variation",
        "GDP_Growth"
        
    ]
]

In [75]:
numeric_cols = final_df3.select_dtypes(include="number").columns

missing_by_country = final_df3.groupby("country")[numeric_cols].apply(
    lambda x: x.isnull().sum().sum()
).reset_index(name="missing_count")

final_df3 = final_df3.merge(missing_by_country, on="country", how="left")
final_df3

,year,country,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment,Inflation_Rate,Unemployment_Variation,Worldwide_Governance_Indicators,Productivity_Variation,GDP_Growth,missing_count
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,27.471257,NaN,NaN,132
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,47.661534,NaN,NaN,138
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,42.081555,NaN,NaN,132
3,2000,ARG,NaN,0.000000,16.711861,34.277230,7.561,64.590212,NaN,-0.788999,48
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,46.781340,NaN,NaN,132
...,...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,69.318,48.902182,34.637478,1.965627,3.697,75.351985,48.513447,1.731397,0
2857,2024,SWE,46.363,27.752405,51.826313,2.835817,8.398,87.089942,332.353013,1.017797,0
2858,2024,TUR,28.557,71.992860,26.944220,58.506450,8.712,37.689790,91.185172,3.327623,0
2859,2024,USA,123.186,0.000000,50.705994,2.949525,4.022,72.373172,90.335094,2.793187,0


In [76]:
final_df3

,year,country,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment,Inflation_Rate,Unemployment_Variation,Worldwide_Governance_Indicators,Productivity_Variation,GDP_Growth,missing_count
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,27.471257,NaN,NaN,132
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,47.661534,NaN,NaN,138
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,42.081555,NaN,NaN,132
3,2000,ARG,NaN,0.000000,16.711861,34.277230,7.561,64.590212,NaN,-0.788999,48
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,46.781340,NaN,NaN,132
...,...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,69.318,48.902182,34.637478,1.965627,3.697,75.351985,48.513447,1.731397,0
2857,2024,SWE,46.363,27.752405,51.826313,2.835817,8.398,87.089942,332.353013,1.017797,0
2858,2024,TUR,28.557,71.992860,26.944220,58.506450,8.712,37.689790,91.185172,3.327623,0
2859,2024,USA,123.186,0.000000,50.705994,2.949525,4.022,72.373172,90.335094,2.793187,0


In [77]:
final_df3

,year,country,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment,Inflation_Rate,Unemployment_Variation,Worldwide_Governance_Indicators,Productivity_Variation,GDP_Growth,missing_count
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,27.471257,NaN,NaN,132
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,47.661534,NaN,NaN,138
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,42.081555,NaN,NaN,132
3,2000,ARG,NaN,0.000000,16.711861,34.277230,7.561,64.590212,NaN,-0.788999,48
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,46.781340,NaN,NaN,132
...,...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,69.318,48.902182,34.637478,1.965627,3.697,75.351985,48.513447,1.731397,0
2857,2024,SWE,46.363,27.752405,51.826313,2.835817,8.398,87.089942,332.353013,1.017797,0
2858,2024,TUR,28.557,71.992860,26.944220,58.506450,8.712,37.689790,91.185172,3.327623,0
2859,2024,USA,123.186,0.000000,50.705994,2.949525,4.022,72.373172,90.335094,2.793187,0


In [78]:
final_df3 = final_df3[final_df3.isnull().sum(axis=1) <= 6]

In [79]:
final_df3

,year,country,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment,Inflation_Rate,Unemployment_Variation,Worldwide_Governance_Indicators,Productivity_Variation,GDP_Growth,missing_count
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,27.471257,NaN,NaN,132
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,47.661534,NaN,NaN,138
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,42.081555,NaN,NaN,132
3,2000,ARG,NaN,0.000000,16.711861,34.277230,7.561,64.590212,NaN,-0.788999,48
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,46.781340,NaN,NaN,132
...,...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,69.318,48.902182,34.637478,1.965627,3.697,75.351985,48.513447,1.731397,0
2857,2024,SWE,46.363,27.752405,51.826313,2.835817,8.398,87.089942,332.353013,1.017797,0
2858,2024,TUR,28.557,71.992860,26.944220,58.506450,8.712,37.689790,91.185172,3.327623,0
2859,2024,USA,123.186,0.000000,50.705994,2.949525,4.022,72.373172,90.335094,2.793187,0


In [ ]:
# final_dataset_full_clean.csv was a duplicate of the final complete dataset and is no longer exported

In [81]:
final_df1.to_csv(PROCESSED_DIR / "final_dataset_core_indicators.csv", index=False)
final_df2.to_csv(PROCESSED_DIR / "final_dataset_governance_indicators.csv", index=False)

In [82]:
col = "Energy_Import_Dependency"
missing_countries = final_df3.loc[
    final_df3[col].isnull(),
    ["country", col]
].drop_duplicates()

In [83]:
missing_countries

,country,Energy_Import_Dependency


In [ ]:
# Diagnostic export removed; keep the result in memory only

In [85]:
col = "Public_Debt (%GDP)"
missing_countries2 = final_df3.loc[
    final_df3[col].isnull(),
    ["country", col]
].drop_duplicates()
missing_countries2

,country,Public_Debt (%GDP)
0,AGO,NaN
1,ALB,NaN
2,ARE,NaN
3,ARG,NaN
4,ARM,NaN
...,...,...
1438,SSD,NaN
1452,TZA,NaN
1524,LBN,NaN
2079,TGO,NaN


In [ ]:
# Diagnostic export removed; keep the result in memory only

In [87]:
final_df3.to_csv(PROCESSED_DIR / "final_dataset_complete.csv", index=False)

In [88]:
#to find where saved
import os
os.getcwd()

'c:\\Documenti\\UNIMIB\\Data Science Lab\\PROJECT'

In [89]:
import numpy as np

# function to calculate recovery time
def calculate_recovery_time(df, shock_year, gdp_col):

    recovery_times = []

    for country in df["country"].unique():

        country_df = df[df["country"] == country].sort_values("year")

        # get GDP value before shock
        try:
            baseline = country_df.loc[
                country_df["year"] == shock_year,
                gdp_col
            ].values[0]
        except:
            continue

        recovery_year = np.nan

        # search future years
        future_data = country_df[country_df["year"] > shock_year]

        for _, row in future_data.iterrows():

            if row[gdp_col] >= baseline:
                recovery_year = row["year"] - shock_year
                break

        # assign result to all rows of that country
        country_df = country_df.copy()
        country_df[f"recovery_{shock_year}"] = recovery_year

        recovery_times.append(country_df)

    return pd.concat(recovery_times)


# =========================
# APPLY FOR 2007
# =========================
final_df2 = calculate_recovery_time(
    final_df2,
    shock_year=2007,
    gdp_col="GDP_Growth"
)

# =========================
# APPLY FOR 2019
# =========================
final_df2 = calculate_recovery_time(
    final_df2,
    shock_year=2019,
    gdp_col="GDP_Growth"
)

In [90]:
final_df2.head(100)

,year,country,Inflation_Rate,Unemployment_Variation,Worldwide_Governance_Indicators,Productivity_Variation,GDP_Growth,recovery_2007,recovery_2019
0,2000,AGO,NaN,NaN,27.471257,NaN,NaN,NaN,NaN
120,2002,AGO,NaN,NaN,32.614381,NaN,NaN,NaN,NaN
240,2003,AGO,NaN,NaN,33.925372,NaN,NaN,NaN,NaN
360,2004,AGO,NaN,NaN,33.803630,NaN,NaN,NaN,NaN
482,2005,AGO,NaN,NaN,35.692131,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
486,2005,ARM,NaN,NaN,46.912304,NaN,NaN,NaN,NaN
607,2006,ARM,NaN,NaN,47.529747,NaN,NaN,NaN,NaN
728,2007,ARM,NaN,NaN,48.955239,NaN,NaN,NaN,NaN
849,2008,ARM,NaN,NaN,44.656769,NaN,NaN,NaN,NaN


In [ ]:
# Duplicate final exports removed; keep only the final named outputs below

In [ ]:
from pathlib import Path

REPORT_DIR = Path("Data") / "Processed" / "missing_value_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def export_missing_value_reports(df, dataset_name, country_col="country", output_dir=REPORT_DIR):
    """Create country-by-column missing-value reports and return them in memory only."""
    if country_col not in df.columns:
        raise KeyError(f"{dataset_name}: missing required column '{country_col}'")

    working = df.copy()
    working["_country_report"] = working[country_col].fillna("MISSING_COUNTRY")
    feature_columns = [column for column in working.columns if column not in {"_country_report", country_col}]

    missing_by_country_wide = (
        working.groupby("_country_report", dropna=False)[feature_columns]
        .agg(lambda series: int(series.isna().sum()))
        .reset_index()
        .rename(columns={"_country_report": "country"})
    )

    missing_by_country_long = (
        missing_by_country_wide
        .melt(id_vars=["country"], var_name="column", value_name="missing_cells")
        .sort_values(["country", "column"])
        .reset_index(drop=True)
    )
    missing_by_country_long["missing_cells"] = missing_by_country_long["missing_cells"].astype(int)

    missing_by_column = (
        pd.DataFrame({
            "column": df.columns,
            "missing_count": df.isna().sum().astype(int).to_numpy(),
            "missing_share": df.isna().mean().to_numpy(),
        })
        .sort_values(["missing_count", "column"], ascending=[False, True])
        .reset_index(drop=True)
    )

    country_summary = (
        missing_by_country_long
        .groupby("country", as_index=False)["missing_cells"]
        .sum()
        .rename(columns={"missing_cells": "total_missing_cells"})
        .sort_values(["total_missing_cells", "country"], ascending=[False, True])
        .reset_index(drop=True)
    )

    row_summary = (
        working.assign(has_missing=working.isna().any(axis=1))
        .groupby("_country_report", dropna=False)
        .agg(
            rows=("has_missing", "size"),
            rows_with_any_missing=("has_missing", "sum"),
        )
        .reset_index()
        .rename(columns={"_country_report": "country"})
    )

    country_summary = country_summary.merge(row_summary, on="country", how="left")
    country_summary["rows"] = country_summary["rows"].fillna(0).astype(int)
    country_summary["rows_with_any_missing"] = country_summary["rows_with_any_missing"].fillna(0).astype(int)

    rows_with_missing_country = df[df[country_col].isna()].copy()
    rows_with_any_missing = df[df.isna().any(axis=1)].copy()

    total_missing_cells = int(df.isna().sum().sum())
    print(f"[{dataset_name}] total missing cells: {total_missing_cells}")
    print(f"[{dataset_name}] rows with missing country labels: {len(rows_with_missing_country)}")
    print(f"[{dataset_name}] report kept in memory only")

    return {
        "missing_by_country_long": missing_by_country_long,
        "missing_by_country_wide": missing_by_country_wide,
        "missing_by_column": missing_by_column,
        "country_summary": country_summary,
        "rows_with_missing_country": rows_with_missing_country,
        "rows_with_any_missing": rows_with_any_missing,
    }


reports = {}

for dataset_name in ["final_clean", "final_merged", "final_df3"]:
    if dataset_name in globals():
        reports[dataset_name] = export_missing_value_reports(globals()[dataset_name], dataset_name)

if "final_clean" in reports:
    display(reports["final_clean"]["country_summary"].head(25))
    display(reports["final_clean"]["missing_by_column"])

[final_clean] total missing cells: 20249
[final_clean] rows with missing country labels: 0
[final_clean] report exported to: Data\Processed\missing_value_reports
[final_merged] total missing cells: 14666
[final_merged] rows with missing country labels: 0
[final_merged] report exported to: Data\Processed\missing_value_reports
[final_df3] total missing cells: 10645
[final_df3] rows with missing country labels: 0
[final_df3] report exported to: Data\Processed\missing_value_reports


,country,total_missing_cells,rows,rows_with_any_missing
0,EU,208,26,26
1,MISSING_COUNTRY,208,26,26
2,USMCA,208,26,26
3,EU19OECD,200,25,25
4,EU22OECD,200,25,25
5,EU27,200,25,25
6,TWN,200,25,25
7,ALB,192,24,24
8,AZE,192,24,24
9,BIH,192,24,24


,column,missing_count,missing_share
0,Eurostat_Public_Debt_GDP_2000_2025,3013,0.811036
1,OECD_Productivity_Variation_2000_2025,2571,0.692059
2,Global_Public_Debt_GDP_2000_2025,2528,0.680485
3,OECD_RD_GDP_2000_2025,2447,0.658681
4,OECD_Tertiary_Education_2000_2025,2407,0.647914
5,OECD_GDP_Growth_2000_2025,2366,0.636878
6,OECD_Inflation_CPI_2000_2025,2314,0.622880
7,OECD_Unemployment_Rate_2000_2025,2194,0.590579
8,Global_Energy_Dependency_0_100,409,0.110094
9,country,0,0.000000
